In [ ]:
import pandas as pd
from scipy.stats import ttest_rel
import matplotlib.pyplot as plt
# ========================
# AML
# ========================

AML_with_context_semantic_path = "/Users/justin.seby/Documents/venv/Justin/Karolinska Institutet/DDLS/DDLS Code/DDLS Projects/LADDER Code Repo/Ladder Annotation and Validation/Benchmarking/AML/Intermediate Files/AMLWITHCONTEXT_Semantic_results_general_only_withConfidence.csv"
AML_with_context_roc_path = "/Users/justin.seby/Documents/venv/Justin/Karolinska Institutet/DDLS/DDLS Code/DDLS Projects/LADDER Code Repo/Ladder Annotation and Validation/Benchmarking/AML/Intermediate Files/AMLrouge_aml_results/AMLWithContext_Combined_Annotations_all_rouge_per_geneset_withConfidence.csv"

AML_without_context_semantic_path = "/Users/justin.seby/Documents/venv/Justin/Karolinska Institutet/DDLS/DDLS Code/DDLS Projects/LADDER Code Repo/Ladder Annotation and Validation/Benchmarking/AML/Intermediate Files/AMLWITHOUTCONTEXT_Semantic_results_general_only_withConfidence.csv"
AML_without_context_roc_path = "/Users/justin.seby/Documents/venv/Justin/Karolinska Institutet/DDLS/DDLS Code/DDLS Projects/LADDER Code Repo/Ladder Annotation and Validation/Benchmarking/AML/Intermediate Files/AMLrouge_aml_results/AMLWithoutContext_Combined_Annotations_all_rouge_per_geneset_withConfidence.csv"


# ========================
# Breast cancer
# ========================




BREAST_with_context_semantic_path = "/Users/justin.seby/Documents/venv/Justin/Karolinska Institutet/DDLS/DDLS Code/DDLS Projects/LADDER Code Repo/Ladder Annotation and Validation/Benchmarking/Breast Cancer/Intermediate Files/BREASTWITHCONTEXT_Semantic_results_general_only_withConfidence.csv"
BREAST_with_context_roc_path = "/Users/justin.seby/Documents/venv/Justin/Karolinska Institutet/DDLS/DDLS Code/DDLS Projects/LADDER Code Repo/Ladder Annotation and Validation/Benchmarking/Breast Cancer/Intermediate Files/Breastrouge__results/BreastCancerWithContext_Combined_Annotations_all_rouge_per_geneset_withConfidence.csv"

BREAST_without_context_semantic_path = "/Users/justin.seby/Documents/venv/Justin/Karolinska Institutet/DDLS/DDLS Code/DDLS Projects/LADDER Code Repo/Ladder Annotation and Validation/Benchmarking/Breast Cancer/Intermediate Files/BREASTWITHOUTCONTEXT_Semantic_results_general_only_withConfidence.csv"
BREAST_without_context_roc_path = "/Users/justin.seby/Documents/venv/Justin/Karolinska Institutet/DDLS/DDLS Code/DDLS Projects/LADDER Code Repo/Ladder Annotation and Validation/Benchmarking/Breast Cancer/Intermediate Files/Breastrouge__results/BreastCancerWithoutContext_Combined_Annotations_all_rouge_per_geneset_withConfidence.csv"


# ========================
# Lung cancer
# ========================






LUNG_with_context_semantic_path = "/Users/justin.seby/Documents/venv/Justin/Karolinska Institutet/DDLS/DDLS Code/DDLS Projects/LADDER Code Repo/Ladder Annotation and Validation/Benchmarking/Lung Cancer/Intermediate Files/LUNGWITHCONTEXT_Semantic_results_general_only_withConfidence.csv"
LUNG_with_context_roc_path = "/Users/justin.seby/Documents/venv/Justin/Karolinska Institutet/DDLS/DDLS Code/DDLS Projects/LADDER Code Repo/Ladder Annotation and Validation/Benchmarking/Lung Cancer/Intermediate Files/LUNGrouge_results/LungWithContext_Combined_Annotations_all_rouge_per_geneset_withConfidence.csv"

LUNG_without_context_semantic_path = "/Users/justin.seby/Documents/venv/Justin/Karolinska Institutet/DDLS/DDLS Code/DDLS Projects/LADDER Code Repo/Ladder Annotation and Validation/Benchmarking/Lung Cancer/Intermediate Files/LUNGWITHOUTCONTEXT_Semantic_results_general_only_withConfidence.csv"
LUNG_without_context_roc_path = "/Users/justin.seby/Documents/venv/Justin/Karolinska Institutet/DDLS/DDLS Code/DDLS Projects/LADDER Code Repo/Ladder Annotation and Validation/Benchmarking/Lung Cancer/Intermediate Files/LUNGrouge_results/LungWithoutContext_Combined_Annotations_all_rouge_per_geneset_withConfidence.csv"

In [ ]:
# ========================
# COLLECT ALL PATHS
# ========================

paths = {

    ("AML", "with_context", "semantic"): AML_with_context_semantic_path,
    ("AML", "with_context", "roc"): AML_with_context_roc_path,
    ("AML", "without_context", "semantic"): AML_without_context_semantic_path,
    ("AML", "without_context", "roc"): AML_without_context_roc_path,

    ("BREAST", "with_context", "semantic"): BREAST_with_context_semantic_path,
    ("BREAST", "with_context", "roc"): BREAST_with_context_roc_path,
    ("BREAST", "without_context", "semantic"): BREAST_without_context_semantic_path,
    ("BREAST", "without_context", "roc"): BREAST_without_context_roc_path,

    ("LUNG", "with_context", "semantic"): LUNG_with_context_semantic_path,
    ("LUNG", "with_context", "roc"): LUNG_with_context_roc_path,
    ("LUNG", "without_context", "semantic"): LUNG_without_context_semantic_path,
    ("LUNG", "without_context", "roc"): LUNG_without_context_roc_path,
}

In [ ]:
def run_stats(file_path, disease, context, metric_type):

    df = pd.read_csv(file_path)

    print("Running file:", file_path)

    results = []


    # ======================
    # SEMANTIC FILE
    # ======================

    if "LADDER_Similarity" in df.columns:

        ladder_col = "LADDER_Similarity"
        hu_col = "Hu_Similarity"
        ga_col = "GeneAgent_Similarity"
        model_col = "Model"

        models = df[model_col].unique()

        for model in models:

            sub = df[df[model_col] == model].dropna(
                subset=[ladder_col, hu_col, ga_col]
            )

            if len(sub) < 2:
                continue

            diff_hu = (sub[ladder_col] - sub[hu_col]).mean()
            diff_ga = (sub[ladder_col] - sub[ga_col]).mean()

            t_hu, p_hu = ttest_rel(
                sub[ladder_col],
                sub[hu_col],
            )

            t_ga, p_ga = ttest_rel(
                sub[ladder_col],
                sub[ga_col],
            )

            results.append({

                "Disease": disease,
                "Context": context,
                "Metric": "semantic",
                "Model": model,

                "MeanDiff_LADDER_minus_Hu": diff_hu,
                "MeanDiff_LADDER_minus_GeneAgent": diff_ga,

                "t_LADDER_vs_Hu": t_hu,
                "p_LADDER_vs_Hu": p_hu,
                "Hu_significant": p_hu < 0.05,

                "t_LADDER_vs_GeneAgent": t_ga,
                "p_LADDER_vs_GeneAgent": p_ga,
                "GeneAgent_significant": p_ga < 0.05,
            })


    # ======================
    # ROUGE FILE
    # ======================

    elif "Our_rouge1_f" in df.columns:

        df["_Model"] = "ALL"
        model_col = "_Model"

        rouge_metrics = {
            "rouge1": (
                "Our_rouge1_f",
                "Hu_rouge1_f",
                "GeneAgent_rouge1_f",
            ),
            "rouge2": (
                "Our_rouge2_f",
                "Hu_rouge2_f",
                "GeneAgent_rouge2_f",
            ),
            "rougeL": (
                "Our_rougeL_f",
                "Hu_rougeL_f",
                "GeneAgent_rougeL_f",
            ),
        }

        for rouge_name, cols in rouge_metrics.items():

            ladder_col, hu_col, ga_col = cols

            sub = df.dropna(
                subset=[ladder_col, hu_col, ga_col]
            )

            if len(sub) < 2:
                continue

            diff_hu = (sub[ladder_col] - sub[hu_col]).mean()
            diff_ga = (sub[ladder_col] - sub[ga_col]).mean()

            t_hu, p_hu = ttest_rel(
                sub[ladder_col],
                sub[hu_col],
            )

            t_ga, p_ga = ttest_rel(
                sub[ladder_col],
                sub[ga_col],
            )

            results.append({

                "Disease": disease,
                "Context": context,
                "Metric": rouge_name,
                "Model": "ALL",

                "MeanDiff_LADDER_minus_Hu": diff_hu,
                "MeanDiff_LADDER_minus_GeneAgent": diff_ga,

                "t_LADDER_vs_Hu": t_hu,
                "p_LADDER_vs_Hu": p_hu,
                "Hu_significant": p_hu < 0.05,

                "t_LADDER_vs_GeneAgent": t_ga,
                "p_LADDER_vs_GeneAgent": p_ga,
                "GeneAgent_significant": p_ga < 0.05,
            })


    else:
        raise ValueError("Unknown format")


    return results

In [ ]:
all_results = []

for (disease, context, metric), path in paths.items():

    print("Running:", disease, context, metric)

    res = run_stats(path, disease, context, metric)

    all_results.extend(res)


results_df = pd.DataFrame(all_results)

print(results_df)

results_df.to_csv(
    "/Users/justin.seby/Documents/venv/Justin/Karolinska Institutet/DDLS/DDLS Code/DDLS Projects/LADDER Code Repo/Ladder Annotation and Validation/Benchmarking/ALL_STAT_TESTS_RESULTS.csv",
    index=False
)

print("Saved ALL_STAT_TESTS_RESULTS.csv")

In [ ]:
semantic_paths = []
rouge_paths = []

for (disease, context, metric), path in paths.items():

    if metric == "semantic":
        semantic_paths.append((disease, context, path))

    if metric == "roc":
        rouge_paths.append((disease, context, path))

In [ ]:
semantic_list = []

for disease, context, path in semantic_paths:

    df = pd.read_csv(path)

    df["Disease"] = disease
    df["Context"] = context

    semantic_list.append(df)

semantic_all = pd.concat(semantic_list, ignore_index=True)

print("Semantic rows:", len(semantic_all))


In [ ]:
rouge_list = []

for disease, context, path in rouge_paths:

    df = pd.read_csv(path)

    df["Disease"] = disease
    df["Context"] = context

    rouge_list.append(df)

rouge_all = pd.concat(rouge_list, ignore_index=True)

print("Rouge rows:", len(rouge_all))

In [ ]:
from scipy.stats import ttest_rel
import pandas as pd

semantic_results = []

for context in semantic_all["Context"].unique():

    sub_ctx = semantic_all[semantic_all["Context"] == context]

    for model in sub_ctx["Model"].unique():

        sub = sub_ctx[sub_ctx["Model"] == model].dropna(
            subset=[
                "LADDER_Similarity",
                "Hu_Similarity",
                "GeneAgent_Similarity",
            ]
        )

        if len(sub) < 2:
            continue

        t_hu, p_hu = ttest_rel(
            sub["LADDER_Similarity"],
            sub["Hu_Similarity"],
        )

        t_ga, p_ga = ttest_rel(
            sub["LADDER_Similarity"],
            sub["GeneAgent_Similarity"],
        )

        semantic_results.append({

            "Context": context,
            "Model": model,
            "Metric": "semantic",

            "p_Hu": p_hu,
            "Hu_significant": p_hu < 0.05,

            "p_GeneAgent": p_ga,
            "GeneAgent_significant": p_ga < 0.05,
        })


semantic_results_df = pd.DataFrame(semantic_results)

semantic_results_df.to_csv(
    "COMBINED_semantic_stats.csv",
    index=False
)

print("Saved COMBINED_semantic_stats.csv")

In [ ]:
rouge_results = []

metrics = {
    "rouge1": (
        "Our_rouge1_f",
        "Hu_rouge1_f",
        "GeneAgent_rouge1_f",
    ),
    "rouge2": (
        "Our_rouge2_f",
        "Hu_rouge2_f",
        "GeneAgent_rouge2_f",
    ),
    "rougeL": (
        "Our_rougeL_f",
        "Hu_rougeL_f",
        "GeneAgent_rougeL_f",
    ),
}


for context in rouge_all["Context"].unique():

    sub_ctx = rouge_all[rouge_all["Context"] == context]

    for metric_name, cols in metrics.items():

        ladder_col, hu_col, ga_col = cols

        sub = sub_ctx.dropna(
            subset=[ladder_col, hu_col, ga_col]
        )

        if len(sub) < 2:
            continue

        t_hu, p_hu = ttest_rel(
            sub[ladder_col],
            sub[hu_col],
        )

        t_ga, p_ga = ttest_rel(
            sub[ladder_col],
            sub[ga_col],
        )

        rouge_results.append({

            "Context": context,
            "Metric": metric_name,

            "p_Hu": p_hu,
            "Hu_significant": p_hu < 0.05,

            "p_GeneAgent": p_ga,
            "GeneAgent_significant": p_ga < 0.05,
        })


rouge_results_df = pd.DataFrame(rouge_results)

rouge_results_df.to_csv(
    "COMBINED_rouge_stats.csv",
    index=False
)

print("Saved COMBINED_rouge_stats.csv")